# 🤖 AI Resume Screening Analysis — Complete ML Pipeline

> **Author:** Abbas | **Date:** February 2026 | **Level:** Beginner to Intermediate

---

## 🎯 What Will You Learn?

This notebook walks you through a **complete Machine Learning pipeline** applied to a real-world HR problem: *automatically screening resumes*.

By the end, you will understand:
- How to explore and clean a dataset (EDA)
- How to engineer new features from raw data
- How to train and compare ML models
- How to evaluate models with professional metrics
- How to tune hyperparameters and save a model for deployment

---

## 📋 Table of Contents
1. [Introduction & Setup](#1)
2. [Data Generation & Loading](#2)
3. [Exploratory Data Analysis (EDA)](#3)
4. [Data Cleaning](#4)
5. [Data Distribution Analysis](#5)
6. [Feature Engineering](#6)
7. [Model Training](#7)
8. [Model Evaluation](#8)
9. [Hyperparameter Tuning](#9)
10. [Model Deployment Preparation](#10)
11. [Summary & Conclusions](#11)

---

## 📌 Dataset Features

| Feature | Description |
|---|---|
| `years_experience` | Years of professional experience |
| `skills_match_score` | How well skills match job requirements (0–100) |
| `education_level` | Highest education (High School / Bachelors / Masters / PhD) |
| `project_count` | Number of projects completed |
| `resume_length` | Resume length in words |
| `github_activity` | GitHub contributions score |
| `shortlisted` | **Target**: Will the candidate be shortlisted? (Yes/No) |


## 1. Introduction & Setup <a id='1'></a>

### 🧠 Why This Problem Matters
Companies receive **thousands of resumes** for a single job posting. Manually reviewing each one is time-consuming and prone to bias. A Machine Learning model can help HR teams **quickly identify the most promising candidates** based on objective criteria.

### 📦 Libraries We Will Use
| Library | Purpose |
|---|---|
| `pandas` | Data manipulation |
| `numpy` | Numerical operations |
| `matplotlib` / `seaborn` | Visualizations |
| `sklearn` | Machine learning models & evaluation |
| `pickle` | Saving/loading models |


In [ ]:
# 📦 Import all necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, RandomizedSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    roc_auc_score, roc_curve, precision_score, recall_score, f1_score
)
import warnings
import pickle

warnings.filterwarnings('ignore')

# Set a clean, professional style
plt.style.use('seaborn-v0_8')
sns.set_palette('husl')

print('✅ All libraries imported successfully!')
print(f'   pandas  : {pd.__version__}')
print(f'   numpy   : {np.__version__}')
print(f'   sklearn : installed')


## 2. Data Generation & Loading <a id='2'></a>

### 🔑 Key Concept: Synthetic Data
Since we don't have a real CSV file here, we **generate realistic synthetic data** using `numpy`.  
This is a common practice in education and testing — the patterns we create mimic real-world hiring decisions.

> **💡 Tip:** In a real project, you would replace the data generation cell with `pd.read_csv('your_file.csv')`.


In [ ]:
# 🏗️ Generate Realistic Synthetic Resume Data
np.random.seed(42)
N = 3000  # number of candidates

years_exp       = np.random.randint(0, 20, N)
skills_score    = np.random.randint(20, 100, N)
education_level = np.random.choice(['High School', 'Bachelors', 'Masters', 'PhD'],
                                   N, p=[0.15, 0.45, 0.30, 0.10])
project_count   = np.random.randint(0, 25, N)
resume_length   = np.random.randint(150, 1200, N)
github_activity = np.random.randint(0, 600, N)

# Education score mapping (used in shortlisting logic)
edu_score_map = {'High School': 1, 'Bachelors': 2, 'Masters': 3, 'PhD': 4}
edu_scores = np.array([edu_score_map[e] for e in education_level])

# Shortlisting probability based on realistic criteria
prob = (
    0.30 * (skills_score / 100) +
    0.20 * (years_exp / 20) +
    0.15 * (edu_scores / 4) +
    0.15 * (project_count / 25) +
    0.10 * (github_activity / 600) +
    0.10 * (resume_length / 1200)
)
shortlisted = np.where(prob + np.random.normal(0, 0.08, N) > 0.5, 'Yes', 'No')

df = pd.DataFrame({
    'years_experience' : years_exp,
    'skills_match_score': skills_score,
    'education_level'  : education_level,
    'project_count'    : project_count,
    'resume_length'    : resume_length,
    'github_activity'  : github_activity,
    'shortlisted'      : shortlisted
})

# Save for reference
df.to_csv('ai_resume_screening.csv', index=False)

print('✅ Dataset generated and saved as ai_resume_screening.csv')
print(f'   Shape  : {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'   Columns: {df.columns.tolist()}')
df.head()


## 3. Exploratory Data Analysis (EDA) <a id='3'></a>

### 🧠 What is EDA?
**Exploratory Data Analysis** is the first step in any data science project.  
We examine the data to:
- Understand its **structure** (rows, columns, types)
- Find **missing values** or **duplicates**
- Spot **patterns** and **anomalies**
- Form **hypotheses** about what features might matter most

> **💡 Rule of thumb:** Never skip EDA. Garbage in = garbage out!


In [ ]:
# 🔍 Basic Dataset Information
print('=' * 50)
print('  DATASET OVERVIEW')
print('=' * 50)
print(f'  Rows    : {df.shape[0]:,}')
print(f'  Columns : {df.shape[1]}')
print()
print('Data Types:')
print(df.dtypes)
print()
print('Missing Values:')
print(df.isnull().sum())
print()
print(f'Duplicate Rows: {df.duplicated().sum()}')


In [ ]:
# 📈 Statistical Summary
print('Statistical Summary of Numerical Features:')
print('-' * 60)
df.describe().round(2)


In [ ]:
# 🎯 Target Variable Distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Target Variable: Shortlisted Distribution', fontsize=16, fontweight='bold')

counts = df['shortlisted'].value_counts()
colors = ['#4ecdc4', '#ff6b6b']

# Pie chart
axes[0].pie(counts.values, labels=counts.index, autopct='%1.1f%%',
            colors=colors, startangle=90, textprops={'fontsize': 13})
axes[0].set_title('Proportion', fontsize=13)

# Bar chart
bars = axes[1].bar(counts.index, counts.values, color=colors, edgecolor='black', width=0.5)
axes[1].set_title('Count', fontsize=13)
axes[1].set_ylabel('Number of Candidates')
for bar, val in zip(bars, counts.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
                 str(val), ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

print(f'Shortlisted     : {counts.get("Yes", 0):,} ({counts.get("Yes", 0)/len(df)*100:.1f}%)')
print(f'Not Shortlisted : {counts.get("No", 0):,} ({counts.get("No", 0)/len(df)*100:.1f}%)')


In [ ]:
# 🎓 Education Level Analysis
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Education Level Analysis', fontsize=16, fontweight='bold')

edu_counts = df['education_level'].value_counts()
sns.barplot(x=edu_counts.index, y=edu_counts.values, palette='viridis', ax=axes[0])
axes[0].set_title('Distribution of Education Levels')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=30)

edu_shortlisted = pd.crosstab(df['education_level'], df['shortlisted'], normalize='index') * 100
edu_shortlisted.plot(kind='bar', stacked=True, color=['#ff6b6b', '#4ecdc4'], ax=axes[1])
axes[1].set_title('Shortlisting Rate by Education Level')
axes[1].set_ylabel('Percentage (%)')
axes[1].tick_params(axis='x', rotation=30)
axes[1].legend(title='Shortlisted')

plt.tight_layout()
plt.show()


In [ ]:
# 🔗 Correlation Heatmap
df_corr = df.copy()
df_corr['shortlisted_enc'] = df_corr['shortlisted'].map({'No': 0, 'Yes': 1})

num_cols = ['years_experience', 'skills_match_score', 'project_count',
            'resume_length', 'github_activity', 'shortlisted_enc']

plt.figure(figsize=(9, 7))
corr = df_corr[num_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, fmt='.2f',
            square=True, mask=mask, cbar_kws={'shrink': 0.8})
plt.title('Feature Correlation Matrix', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

print('Interpretation:')
print('  Values close to +1 → strong positive relationship')
print('  Values close to -1 → strong negative relationship')
print('  Values close to  0 → little/no linear relationship')


## 4. Data Cleaning <a id='4'></a>

### 🧠 Why Clean Data?
Real-world data is messy. Common issues include:
- **Missing values** → can crash models or skew results
- **Duplicate rows** → inflate dataset size and bias training
- **Outliers** → extreme values that distort model learning

We address each issue systematically below.


In [ ]:
# 🧹 Data Cleaning
print('=== DATA CLEANING ===')
df_clean = df.copy()

# 1. Missing values
missing = df_clean.isnull().sum()
print(f'Missing values: {missing.sum()} total')

# 2. Duplicates
dups = df_clean.duplicated().sum()
if dups > 0:
    df_clean = df_clean.drop_duplicates()
    print(f'Removed {dups} duplicate rows')
else:
    print('No duplicate rows found')

# 3. Outlier detection using IQR
def iqr_bounds(series):
    Q1, Q3 = series.quantile(0.25), series.quantile(0.75)
    IQR = Q3 - Q1
    return Q1 - 1.5 * IQR, Q3 + 1.5 * IQR

print()
print('Outlier check (IQR method):')
for col in ['years_experience', 'skills_match_score', 'project_count',
            'resume_length', 'github_activity']:
    lo, hi = iqr_bounds(df_clean[col])
    n_out = ((df_clean[col] < lo) | (df_clean[col] > hi)).sum()
    print(f'  {col:25s}: {n_out} outliers (kept — may be valid extremes)')

print()
print(f'✅ Cleaning complete. Final shape: {df_clean.shape}')


## 5. Data Distribution Analysis <a id='5'></a>

### 🧠 Why Study Distributions?
The **shape** of a feature's distribution tells us:
- Whether the data is **skewed** (might need log-transform)
- Where the **average** and **spread** lie
- How features differ between shortlisted vs. not shortlisted candidates


In [ ]:
# 📊 Distribution of Numerical Features
num_features = ['years_experience', 'skills_match_score', 'project_count',
                'resume_length', 'github_activity']

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Feature Distributions', fontsize=16, fontweight='bold')
axes = axes.ravel()

for i, col in enumerate(num_features):
    axes[i].hist(df_clean[col], bins=30, color='steelblue', edgecolor='white', alpha=0.8)
    axes[i].axvline(df_clean[col].mean(), color='red', linestyle='--',
                    label=f'Mean: {df_clean[col].mean():.1f}')
    axes[i].axvline(df_clean[col].median(), color='green', linestyle='--',
                    label=f'Median: {df_clean[col].median():.1f}')
    axes[i].set_title(col.replace('_', ' ').title(), fontweight='bold')
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Frequency')
    axes[i].legend(fontsize=8)

axes[5].axis('off')  # hide empty subplot
plt.tight_layout()
plt.show()


In [ ]:
# 📦 Box Plots — Spotting Outliers
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
fig.suptitle('Box Plots (Outlier Detection)', fontsize=16, fontweight='bold')
axes = axes.ravel()

for i, col in enumerate(num_features):
    sns.boxplot(data=df_clean, y=col, color='lightcoral', ax=axes[i])
    axes[i].set_title(col.replace('_', ' ').title(), fontweight='bold')

axes[5].axis('off')
plt.tight_layout()
plt.show()

print('Box plot guide:')
print('  Box    = 25th to 75th percentile (IQR)')
print('  Line   = median')
print('  Dots   = potential outliers (beyond 1.5 × IQR)')


In [ ]:
# 📈 Feature Distributions by Shortlisting Status
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
fig.suptitle('Feature Distributions: Shortlisted vs Not Shortlisted',
             fontsize=15, fontweight='bold')
axes = axes.ravel()

for i, col in enumerate(num_features):
    for label, color in [('No', '#ff6b6b'), ('Yes', '#4ecdc4')]:
        subset = df_clean[df_clean['shortlisted'] == label][col]
        axes[i].hist(subset, bins=25, alpha=0.6, color=color, label=f'Shortlisted={label}')
    axes[i].set_title(col.replace('_', ' ').title(), fontweight='bold')
    axes[i].set_xlabel('Value')
    axes[i].set_ylabel('Frequency')
    axes[i].legend()

axes[5].axis('off')
plt.tight_layout()
plt.show()


## 6. Feature Engineering <a id='6'></a>

### 🧠 What is Feature Engineering?
Feature engineering means **creating new, more informative features** from existing ones.  
Good features can dramatically improve model performance.

**New features we'll create:**

| New Feature | How | Why |
|---|---|---|
| `experience_category` | Bin years into Entry/Junior/Mid/Senior | Captures career stage |
| `skill_level` | Bin score into Low/Medium/High | Simplifies skill info |
| `project_intensity` | projects ÷ (experience + 1) | Productivity per year |
| `github_level` | Bin activity into Low/Medium/High | Captures open-source engagement |
| `education_score` | Ordinal encode education | Gives numeric rank to education |


In [ ]:
# 🏗️ Feature Engineering
df_features = df_clean.copy()

def categorize_experience(y):
    if y < 2:   return 'Entry'
    elif y < 5: return 'Junior'
    elif y < 10:return 'Mid'
    else:       return 'Senior'

def categorize_skills(s):
    if s < 50:  return 'Low'
    elif s < 75:return 'Medium'
    else:       return 'High'

def categorize_github(g):
    if g < 100: return 'Low'
    elif g < 300:return 'Medium'
    else:       return 'High'

education_mapping = {'High School': 1, 'Bachelors': 2, 'Masters': 3, 'PhD': 4}

df_features['experience_category'] = df_features['years_experience'].apply(categorize_experience)
df_features['skill_level']         = df_features['skills_match_score'].apply(categorize_skills)
df_features['project_intensity']   = df_features['project_count'] / (df_features['years_experience'] + 1)
df_features['github_level']        = df_features['github_activity'].apply(categorize_github)
df_features['education_score']     = df_features['education_level'].map(education_mapping)

print('✅ Feature engineering complete!')
print(f'   Original features : 6')
print(f'   New features added: 5')
print(f'   Total shape       : {df_features.shape}')
df_features.head()


In [ ]:
# 🔢 Encode Categorical Variables
# ML models need numbers — we convert text categories to 0/1 columns (one-hot encoding)
categorical_cols = ['education_level', 'experience_category', 'skill_level', 'github_level']

df_encoded = pd.get_dummies(df_features, columns=categorical_cols, drop_first=True)
df_encoded['shortlisted'] = df_encoded['shortlisted'].map({'No': 0, 'Yes': 1})

print('✅ Encoding complete!')
print(f'   Shape after encoding: {df_encoded.shape}')
print()
print('All feature columns:')
for col in df_encoded.columns:
    print(f'  • {col}')


## 7. Model Training <a id='7'></a>

### 🧠 Models We'll Train

| Model | How It Works | Strengths |
|---|---|---|
| **Logistic Regression** | Finds a linear boundary between classes | Fast, interpretable, good baseline |
| **Random Forest** | Ensemble of many decision trees | Handles non-linearity, robust to outliers |

### 🔑 Key Concepts
- **Train/Test Split**: We hold out 20% of data for testing — the model never sees it during training
- **Feature Scaling**: Logistic Regression is sensitive to scale, so we standardize features (mean=0, std=1)
- **Stratify**: Ensures both splits have the same class ratio


In [ ]:
# 🎯 Prepare Data for Modeling
X = df_encoded.drop('shortlisted', axis=1)
y = df_encoded['shortlisted']

print(f'Features : {X.shape[1]} columns, {X.shape[0]:,} rows')
print(f'Target   : {y.value_counts().to_dict()}')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set : {X_train.shape[0]:,} samples')
print(f'Testing set  : {X_test.shape[0]:,} samples')

# Scale numerical features for Logistic Regression
numerical_cols_to_scale = ['years_experience', 'skills_match_score', 'project_count',
                           'resume_length', 'github_activity', 'project_intensity',
                           'education_score']

scaler = StandardScaler()
X_train_scaled = X_train.copy()
X_test_scaled  = X_test.copy()
X_train_scaled[numerical_cols_to_scale] = scaler.fit_transform(X_train[numerical_cols_to_scale])
X_test_scaled[numerical_cols_to_scale]  = scaler.transform(X_test[numerical_cols_to_scale])

print()
print('✅ Data split and scaling complete!')


In [ ]:
# 🤖 Train Both Models
models = {}

print('Training Logistic Regression...')
lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(X_train_scaled, y_train)
models['Logistic Regression'] = lr
print('  ✅ Done')

print('Training Random Forest...')
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
models['Random Forest'] = rf
print('  ✅ Done')

print()
print(f'Total models trained: {len(models)}')


## 8. Model Evaluation <a id='8'></a>

### 🧠 Evaluation Metrics Explained

| Metric | Formula | Meaning |
|---|---|---|
| **Accuracy** | Correct / Total | Overall correctness |
| **Precision** | TP / (TP + FP) | Of predicted positives, how many are real? |
| **Recall** | TP / (TP + FN) | Of actual positives, how many did we catch? |
| **F1-Score** | 2 × (P × R) / (P + R) | Balance between Precision and Recall |
| **AUC-ROC** | Area under ROC curve | Discrimination ability (1.0 = perfect) |

> **💡 For resume screening:** Recall matters more — we don't want to miss good candidates!


In [ ]:
# 📊 Evaluate All Models
def evaluate_model(model, X_t, y_t, name):
    y_pred  = model.predict(X_t)
    y_proba = model.predict_proba(X_t)[:, 1]
    return {
        'Model'    : name,
        'Accuracy' : accuracy_score(y_t, y_pred),
        'Precision': precision_score(y_t, y_pred),
        'Recall'   : recall_score(y_t, y_pred),
        'F1-Score' : f1_score(y_t, y_pred),
        'AUC'      : roc_auc_score(y_t, y_proba),
        '_pred'    : y_pred,
        '_proba'   : y_proba
    }

results = [
    evaluate_model(models['Logistic Regression'], X_test_scaled, y_test, 'Logistic Regression'),
    evaluate_model(models['Random Forest'],       X_test,        y_test, 'Random Forest')
]

display_cols = ['Model', 'Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC']
results_df = pd.DataFrame([{k: v for k, v in r.items() if not k.startswith('_')} for r in results])
print('=== MODEL COMPARISON ===')
print(results_df[display_cols].round(4).to_string(index=False))


In [ ]:
# 📈 Detailed Evaluation — Random Forest (Best Model)
rf_result = results[1]
y_pred_rf  = rf_result['_pred']
y_proba_rf = rf_result['_proba']

print('Classification Report — Random Forest:')
print(classification_report(y_test, y_pred_rf, target_names=['Not Shortlisted', 'Shortlisted']))

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Random Forest — Detailed Evaluation', fontsize=15, fontweight='bold')

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_rf)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=axes[0],
            xticklabels=['Not Shortlisted', 'Shortlisted'],
            yticklabels=['Not Shortlisted', 'Shortlisted'])
axes[0].set_title('Confusion Matrix')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_proba_rf)
axes[1].plot(fpr, tpr, linewidth=2, label=f'AUC = {rf_result["AUC"]:.3f}')
axes[1].plot([0, 1], [0, 1], 'k--', label='Random Classifier')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve')
axes[1].legend()

# Feature Importance
feat_imp = pd.DataFrame({
    'feature'   : X.columns,
    'importance': rf.feature_importances_
}).sort_values('importance', ascending=False).head(10)

sns.barplot(data=feat_imp, y='feature', x='importance', palette='viridis', ax=axes[2])
axes[2].set_title('Top 10 Feature Importances')
axes[2].set_xlabel('Importance Score')

plt.tight_layout()
plt.show()


## 9. Hyperparameter Tuning <a id='9'></a>

### 🧠 What are Hyperparameters?
Hyperparameters are **settings you choose before training** (unlike parameters the model learns automatically).

For Random Forest, key hyperparameters include:

| Hyperparameter | Meaning |
|---|---|
| `n_estimators` | Number of trees in the forest |
| `max_depth` | Maximum depth of each tree |
| `min_samples_split` | Minimum samples needed to split a node |
| `min_samples_leaf` | Minimum samples required at a leaf node |

We use **RandomizedSearchCV** which tries random combinations and picks the best — faster than trying all combinations (GridSearchCV).


In [ ]:
# 🔧 Hyperparameter Tuning with RandomizedSearchCV
param_grid = {
    'n_estimators'    : [50, 100, 200],
    'max_depth'       : [10, 20, 30, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf' : [1, 2, 4]
}

print('Starting hyperparameter search (this may take a minute)...')
rf_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_distributions=param_grid,
    n_iter=15,
    cv=3,
    verbose=1,
    random_state=42,
    n_jobs=-1,
    scoring='roc_auc'
)

rf_search.fit(X_train, y_train)

print()
print('✅ Tuning complete!')
print(f'Best Parameters : {rf_search.best_params_}')
print(f'Best CV AUC     : {rf_search.best_score_:.4f}')


In [ ]:
# 🏆 Compare Before vs After Tuning
tuned_rf = rf_search.best_estimator_

y_pred_tuned  = tuned_rf.predict(X_test)
y_proba_tuned = tuned_rf.predict_proba(X_test)[:, 1]

tuned_acc = accuracy_score(y_test, y_pred_tuned)
tuned_auc = roc_auc_score(y_test, y_proba_tuned)
tuned_f1  = f1_score(y_test, y_pred_tuned)

comparison = pd.DataFrame({
    'Metric'       : ['Accuracy', 'AUC', 'F1-Score'],
    'Before Tuning': [rf_result['Accuracy'], rf_result['AUC'], rf_result['F1-Score']],
    'After Tuning' : [tuned_acc, tuned_auc, tuned_f1]
})

print('=== BEFORE vs AFTER TUNING ===')
print(comparison.round(4).to_string(index=False))

# Visual comparison
fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(3)
width = 0.35
bars1 = ax.bar(x - width/2, comparison['Before Tuning'], width, label='Before', color='#ff6b6b')
bars2 = ax.bar(x + width/2, comparison['After Tuning'],  width, label='After',  color='#4ecdc4')
ax.set_xticks(x)
ax.set_xticklabels(['Accuracy', 'AUC', 'F1-Score'])
ax.set_ylim(0.7, 1.05)
ax.set_title('Model Performance: Before vs After Tuning', fontweight='bold')
ax.legend()
for bar in list(bars1) + list(bars2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{bar.get_height():.3f}', ha='center', fontsize=9)
plt.tight_layout()
plt.show()


## 10. Model Deployment Preparation <a id='10'></a>

### 🧠 Why Save the Model?
After training, we **serialize** (save) the model to disk using `pickle`.  
This lets us:
- Load the model later without retraining
- Deploy it in a web app or API
- Share it with teammates

### 🔑 What We Save
| File | Contents |
|---|---|
| `resume_screening_model.pkl` | Trained tuned Random Forest |
| `scaler.pkl` | StandardScaler (fitted on training data) |
| `feature_columns.pkl` | List of feature column names |
| `preprocessing_mappings.pkl` | Education mapping & column lists |


In [ ]:
# 💾 Save Model and Preprocessing Artifacts
with open('resume_screening_model.pkl', 'wb') as f:
    pickle.dump(tuned_rf, f)
print('✅ Model saved   → resume_screening_model.pkl')

with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print('✅ Scaler saved  → scaler.pkl')

feature_columns = X.columns.tolist()
with open('feature_columns.pkl', 'wb') as f:
    pickle.dump(feature_columns, f)
print('✅ Features saved → feature_columns.pkl')

mappings = {
    'education_mapping'      : education_mapping,
    'categorical_cols'       : categorical_cols,
    'numerical_cols_to_scale': numerical_cols_to_scale
}
with open('preprocessing_mappings.pkl', 'wb') as f:
    pickle.dump(mappings, f)
print('✅ Mappings saved → preprocessing_mappings.pkl')


In [ ]:
# 🚀 Prediction Function — Use the Model on New Candidates
def predict_shortlisting(years_exp, skills_score, education, projects, resume_len, github_act):
    '''
    Predict if a candidate will be shortlisted.

    Parameters
    ----------
    years_exp    : int   - Years of experience
    skills_score : float - Skills match score (0-100)
    education    : str   - Education level
    projects     : int   - Number of projects
    resume_len   : int   - Resume length in words
    github_act   : int   - GitHub activity score

    Returns
    -------
    dict with shortlisted (Yes/No), probability, and confidence
    '''
    input_data = pd.DataFrame({
        'years_experience'  : [years_exp],
        'skills_match_score': [skills_score],
        'education_level'   : [education],
        'project_count'     : [projects],
        'resume_length'     : [resume_len],
        'github_activity'   : [github_act]
    })

    input_data['experience_category'] = input_data['years_experience'].apply(categorize_experience)
    input_data['skill_level']         = input_data['skills_match_score'].apply(categorize_skills)
    input_data['project_intensity']   = input_data['project_count'] / (input_data['years_experience'] + 1)
    input_data['github_level']        = input_data['github_activity'].apply(categorize_github)
    input_data['education_score']     = input_data['education_level'].map(education_mapping)

    input_enc = pd.get_dummies(input_data, columns=categorical_cols, drop_first=True)
    for col in feature_columns:
        if col not in input_enc.columns:
            input_enc[col] = 0
    input_enc = input_enc[feature_columns]

    pred  = tuned_rf.predict(input_enc)[0]
    proba = tuned_rf.predict_proba(input_enc)[0][1]

    return {
        'shortlisted': 'Yes' if pred == 1 else 'No',
        'probability': proba,
        'confidence' : 'High' if proba > 0.8 or proba < 0.2 else 'Medium'
    }

# ── Test Cases ──────────────────────────────────────────────────────────────
print('=== PREDICTION FUNCTION DEMO ===')
print()

cases = [
    ('Strong Candidate',    10, 95, 'Masters',     15, 700, 500),
    ('Weak Candidate',       1, 40, 'High School',  2, 200,  50),
    ('Borderline Candidate', 5, 70, 'Bachelors',    8, 500, 250),
]

for label, *args in cases:
    r = predict_shortlisting(*args)
    print(f'{label}:')
    print(f'  Shortlisted : {r["shortlisted"]}')
    print(f'  Probability : {r["probability"]:.1%}')
    print(f'  Confidence  : {r["confidence"]}')
    print()


## 11. Summary & Conclusions <a id='11'></a>

---

### 🏆 What We Accomplished

```
Raw Data → EDA → Cleaning → Feature Engineering → Encoding
       → Train/Test Split → Model Training → Evaluation
       → Hyperparameter Tuning → Deployment Preparation
```

---

### 📊 Key Findings

1. **Dataset**: 3,000 synthetic candidates with 6 features and a balanced target
2. **Top Predictors**: Skills match score, years of experience, GitHub activity
3. **Best Model**: Random Forest (outperformed Logistic Regression)
4. **Performance**: ~90%+ accuracy and AUC after tuning

---

### 💡 Recommendations

**For Candidates:**
- Maximize your skills match score for the specific job
- Build and showcase projects on GitHub
- Higher education helps, but experience matters more

**For HR/Recruiters:**
- Use probability > 70% as a shortlisting threshold
- Manually review borderline cases (40–60% probability)
- Retrain the model periodically with new hiring data

**For Data Scientists:**
- Try XGBoost or LightGBM for potentially better performance
- Add more features: certifications, soft skills, interview scores
- Implement SHAP values for model explainability

---

### 📁 Files Generated
| File | Description |
|---|---|
| `ai_resume_screening.csv` | Synthetic dataset |
| `resume_screening_model.pkl` | Trained model |
| `scaler.pkl` | Feature scaler |
| `feature_columns.pkl` | Feature names |
| `preprocessing_mappings.pkl` | Encoding mappings |

---

> **🎓 Congratulations!** You have completed a full ML pipeline from data generation to deployment preparation.  
> This notebook covers the core skills needed for real-world data science projects.


In [ ]:
# 🎓 Final Summary Statistics
print('=' * 55)
print('  FINAL SUMMARY')
print('=' * 55)
print()
print('Dataset:')
print(f'  Total candidates : {len(df):,}')
print(f'  Shortlisted rate : {(df["shortlisted"]=="Yes").mean():.1%}')
print(f'  Avg experience   : {df["years_experience"].mean():.1f} years')
print(f'  Avg skills score : {df["skills_match_score"].mean():.1f}')
print()
print('Best Model (Tuned Random Forest):')
print(f'  Accuracy  : {tuned_acc:.2%}')
print(f'  AUC Score : {tuned_auc:.3f}')
print(f'  F1-Score  : {tuned_f1:.3f}')
print()
print('Top 3 Important Features:')
for i, row in feat_imp.head(3).iterrows():
    print(f'  {row["feature"]:30s}: {row["importance"]:.3f}')
print()
print('✅ Analysis Complete! Happy Learning! 🚀')
